# Radial Velocity Dispersion Profile

Spatially-resolved stellar kinematics of the AGEL0206 deflector galaxy.

1. Find the deflector center from HST F140W imaging (isophote fitting)
2. Multi-Gaussian expansion (MGE) of the 2D light profile
3. Map IFU spaxels to radial distance from center
4. Bin spaxels by radius (annular rings + PowerBin)
5. Run ppxf on each radial bin → sigma(R) profile
6. Convert to physical units (kpc) and compare to literature

Inspired by Melo-Carneiro et al. (2025) analysis of the Cosmic Horseshoe.

**Target:** AGEL0206 deflector, z = 0.675  
**IFU:** Keck/KCWI medium slicer  
**Imaging:** HST WFC3/IR F140W (0.08"/pix)

## 1. Imports and data loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Circle
from astropy.io import fits
from astropy.wcs import WCS
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
from photutils.isophote import EllipseGeometry, Ellipse as IsoEllipse, build_ellipse_model
from photutils.aperture import EllipticalAperture, CircularAnnulus
import sys
sys.path.insert(0, '..')

from time import perf_counter as clock
from importlib import resources
from urllib import request
from ppxf.ppxf import ppxf
import ppxf.ppxf_util as util
import ppxf.sps_util as lib
from tqdm import tqdm

plt.rcParams['figure.facecolor'] = 'white'
plt.rc('font', family='serif', size=14)
plt.rc('axes', linewidth=1.5, labelsize=16)
plt.rc('xtick', labelsize=14, direction='in')
plt.rc('ytick', labelsize=14, direction='in')

# Cosmology
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
z_defl = 0.675
d_A = cosmo.angular_diameter_distance(z_defl)
kpc_per_arcsec = d_A.to(u.kpc).value * np.pi / 180 / 3600
print(f'Angular diameter distance: {d_A:.1f}')
print(f'Scale: {kpc_per_arcsec:.2f} kpc/arcsec ({1/kpc_per_arcsec:.2f} arcsec/kpc)')

In [ ]:
# Load IFU cube
ifu_file = '../Nov17_2025_DESJ0206_RL_combined_icubes_wcs.fits'
with fits.open(ifu_file) as hdul:
    hdr_ifu = hdul[0].header
    cube = np.asarray(hdul[0].data, dtype=float)

crval = hdr_ifu['CRVAL3']
cdelt = hdr_ifu['CD3_3']
npix = hdr_ifu['NAXIS3']
crpix = hdr_ifu.get('CRPIX3', 1.0)
pix = np.arange(npix)
lam = crval + cdelt * (pix + 1 - crpix)

print(f'IFU cube: {cube.shape}')
print(f'Wavelength: {lam[0]:.0f} - {lam[-1]:.0f} Å')

# Load HST F140W cutout + mask
hst_dir = '../velocity_dispersion_from_IFU' if True else '../../velocity_dispersion_from_IFU'
f140w_file = f'{hst_dir}/AGEL020613-011417A_F140W_WFC3_cutout_L3.fits'
f140w_mask_file = f'{hst_dir}/AGEL020613-011417A_F140W_WFC3_cutout_L3_mask.fits'

with fits.open(f140w_file) as hdul:
    hdr_f140 = hdul[0].header
    img_f140 = hdul[0].data
    wcs_f140 = WCS(hdr_f140)

try:
    with fits.open(f140w_mask_file) as hdul:
        mask_f140 = hdul[0].data.astype(bool)
    print(f'F140W mask loaded: {np.sum(mask_f140)} masked pixels')
except FileNotFoundError:
    mask_f140 = np.zeros_like(img_f140, dtype=bool)
    print('No mask file found — using no mask')

print(f'F140W image: {img_f140.shape}, pixel scale: ~0.08 arcsec/pix')

## 2. Find deflector center via isophote fitting

Use photutils `Ellipse` to fit isophotes to the F140W cutout image.
The center of the innermost isophote gives the precise RA/Dec of the
deflector galaxy center.

In [ ]:
# Initial center guess from the image peak (masked)
img_masked = np.where(~mask_f140, img_f140, 0)
img_masked = np.nan_to_num(img_masked, nan=0.0)

# Smooth slightly to avoid noise peaks
from scipy.ndimage import gaussian_filter
img_smooth = gaussian_filter(img_masked, sigma=2)
y0, x0 = np.unravel_index(np.argmax(img_smooth), img_smooth.shape)
print(f'Initial center guess (pixel): x={x0}, y={y0}')

# Convert to RA/Dec
ra0, dec0 = wcs_f140.pixel_to_world_values(x0, y0)
print(f'Initial center (RA, Dec): {ra0:.6f}, {dec0:.6f}')

# Set up isophote fitting
geometry = EllipseGeometry(x0=x0, y0=y0, sma=10, eps=0.2, pa=0)
ellipse = IsoEllipse(img_masked, geometry)

# Fit isophotes from sma=5 to 50 pixels
try:
    isolist = ellipse.fit_image(maxsma=50, step=0.1, fix_center=False)
    print(f'Fit {len(isolist)} isophotes, sma range: {isolist.sma.min():.1f} - {isolist.sma.max():.1f} pix')
except Exception as e:
    print(f'Isophote fitting failed: {e}')
    print('Trying with fixed center...')
    isolist = ellipse.fit_image(maxsma=50, step=0.1, fix_center=True)

In [ ]:
# Extract fitted center, ellipticity, PA from inner isophotes
inner = isolist.sma < 15  # inner ~1.2 arcsec
x_center = np.median(isolist.x0[inner])
y_center = np.median(isolist.y0[inner])
eps_med = np.median(isolist.eps[inner])
pa_med = np.median(isolist.pa[inner])

print(f'Fitted center (pixel): x={x_center:.1f}, y={y_center:.1f}')
print(f'Median ellipticity: {eps_med:.3f}')
print(f'Median PA: {np.degrees(pa_med):.1f} deg')

# Convert to RA/Dec
ra_center, dec_center = wcs_f140.pixel_to_world_values(x_center, y_center)
print(f'Deflector center: RA={ra_center:.6f}, Dec={dec_center:.6f}')
print(f'Offset from nominal: dRA={3600*(ra_center - 31.55611):.2f}", dDec={3600*(dec_center - (-1.23817)):.2f}"')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Image with isophotes
ax = axes[0]
vmin, vmax = np.nanpercentile(img_f140[img_f140 > 0], [1, 99])
ax.imshow(img_masked, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
ax.plot(x_center, y_center, 'r+', markersize=15, markeredgewidth=2)
# Overlay some isophotes
for iso in isolist[::5]:
    aper = EllipticalAperture((iso.x0, iso.y0), iso.sma, iso.sma*(1-iso.eps), iso.pa)
    aper.plot(ax=ax, color='white', lw=0.5)
ax.set_title('F140W with isophotes')
ax.set_xlim(x_center - 40, x_center + 40)
ax.set_ylim(y_center - 40, y_center + 40)

# Radial profiles
ax = axes[1]
ax.plot(isolist.sma * 0.08, isolist.intens, 'k-o', markersize=3)
ax.set_xlabel('Semi-major axis (arcsec)')
ax.set_ylabel('Mean intensity')
ax.set_title('Isophote intensity profile')
ax.set_yscale('log')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. MGE decomposition from isophote profile

Decompose the 1D intensity profile into a sum of Gaussians.
This gives a smooth parametric model of the light distribution.

In [ ]:
# Fit the isophote intensity profile as a sum of Gaussians
# I(R) = sum_i A_i * exp(-R^2 / (2 * sigma_i^2))
from scipy.optimize import curve_fit

def multi_gaussian_1d(r, *params):
    """Sum of N Gaussians: params = [A1, sigma1, A2, sigma2, ...]"""
    n_gauss = len(params) // 2
    result = np.zeros_like(r)
    for i in range(n_gauss):
        A = params[2*i]
        sigma = params[2*i + 1]
        result += A * np.exp(-r**2 / (2 * sigma**2))
    return result

# Use isophote data (SMA in arcsec, intensity)
r_arcsec = isolist.sma * 0.08  # pixel to arcsec
intensity = isolist.intens
valid = np.isfinite(intensity) & (intensity > 0) & (r_arcsec > 0)
r_fit = r_arcsec[valid]
I_fit = intensity[valid]

# Try 3-Gaussian fit
n_gauss = 3
p0 = []
for i in range(n_gauss):
    p0.extend([I_fit[0] * 0.5**(i), 0.3 * (3**i)])  # decreasing amplitude, increasing width

bounds_lo = [0] * (2 * n_gauss)
bounds_hi = [np.inf, 10.0] * n_gauss  # sigma max = 10 arcsec

try:
    popt, pcov = curve_fit(multi_gaussian_1d, r_fit, I_fit, p0=p0,
                            bounds=(bounds_lo, bounds_hi), maxfev=10000)
    
    print(f'MGE fit ({n_gauss} Gaussians):')
    for i in range(n_gauss):
        print(f'  Component {i+1}: A={popt[2*i]:.2f}, sigma={popt[2*i+1]:.3f} arcsec ({popt[2*i+1]*kpc_per_arcsec:.2f} kpc)')
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(r_fit, I_fit, c='k', s=10, label='Isophote data')
    r_model = np.linspace(0, r_fit.max(), 200)
    ax.plot(r_model, multi_gaussian_1d(r_model, *popt), 'r-', lw=2, label=f'{n_gauss}-Gaussian MGE')
    for i in range(n_gauss):
        comp = popt[2*i] * np.exp(-r_model**2 / (2 * popt[2*i+1]**2))
        ax.plot(r_model, comp, '--', lw=1, alpha=0.7, label=f'Component {i+1}')
    ax.set_xlabel('R (arcsec)')
    ax.set_ylabel('Intensity')
    ax.set_yscale('log')
    ax.legend()
    ax.set_title('Multi-Gaussian Expansion of F140W light profile')
    ax.grid(alpha=0.3)
    plt.show()
    
except Exception as e:
    print(f'MGE fit failed: {e}')
    popt = None

## 4. Map IFU spaxels to radial distance

Use the IFU cube WCS to compute the RA/Dec of each spaxel center,
then calculate angular and physical distance from the deflector center.

In [ ]:
# IFU spatial WCS (axes 1 and 2 of the 3D cube)
wcs_ifu = WCS(hdr_ifu, naxis=2)

# Deflector spaxel region
y_slice = slice(45, 64)  # 19 spaxels
x_slice = slice(45, 55)  # 10 spaxels
ny, nx = 19, 10

# Compute RA/Dec for each spaxel center
yy, xx = np.mgrid[y_slice.start:y_slice.stop, x_slice.start:x_slice.stop]
ra_spax, dec_spax = wcs_ifu.pixel_to_world_values(xx.ravel(), yy.ravel())
ra_spax = ra_spax.reshape(ny, nx)
dec_spax = dec_spax.reshape(ny, nx)

# Angular distance from deflector center (arcsec)
dra = (ra_spax - ra_center) * np.cos(np.radians(dec_center)) * 3600  # arcsec
ddec = (dec_spax - dec_center) * 3600  # arcsec
r_arcsec_spax = np.sqrt(dra**2 + ddec**2)
r_kpc_spax = r_arcsec_spax * kpc_per_arcsec

print(f'Spaxel grid: {ny} x {nx}')
print(f'Radial range: {r_arcsec_spax.min():.2f} - {r_arcsec_spax.max():.2f} arcsec')
print(f'              {r_kpc_spax.min():.1f} - {r_kpc_spax.max():.1f} kpc')

# Plot spaxel positions and radial distances
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# White-light with radial distance overlay
whitelight = np.sum(cube[:, y_slice, x_slice], axis=0)
im = axes[0].imshow(r_arcsec_spax, origin='lower', cmap='RdYlBu_r')
axes[0].set_title('Radial distance from center (arcsec)')
axes[0].set_xlabel('X spaxel')
axes[0].set_ylabel('Y spaxel')
plt.colorbar(im, ax=axes[0], label='R (arcsec)')

# Histogram of radial distances
axes[1].hist(r_arcsec_spax.ravel(), bins=20, edgecolor='k', alpha=0.7)
axes[1].set_xlabel('R (arcsec)')
axes[1].set_ylabel('Number of spaxels')
axes[1].set_title('Distribution of spaxel radial distances')
axes[1].axvline(np.median(r_arcsec_spax), color='r', ls='--', label=f'Median: {np.median(r_arcsec_spax):.2f}"')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5a. Radial binning: annular rings

Average spaxel spectra within concentric annuli at fixed arcsec intervals.

In [ ]:
# Define annular bins
r_edges = np.arange(0, r_arcsec_spax.max() + 0.5, 0.5)  # 0.5 arcsec bins
n_annuli = len(r_edges) - 1

# Extract spectra for each annulus
annular_spectra = []
annular_r_mid = []
annular_n_spax = []

# Noise from sky region
noise_sky = np.std(cube[:, 28:40, 45:70], axis=(1, 2))

for i in range(n_annuli):
    r_lo, r_hi = r_edges[i], r_edges[i+1]
    mask = (r_arcsec_spax >= r_lo) & (r_arcsec_spax < r_hi)
    n_in = np.sum(mask)
    
    if n_in < 2:
        continue
    
    # Average spectra of spaxels in this annulus
    # mask is (ny, nx), cube deflector region is (nwave, ny, nx)
    defl_cube = cube[:, y_slice, x_slice]
    spec_sum = np.average(defl_cube[:, mask], axis=1)
    
    annular_spectra.append(spec_sum)
    annular_r_mid.append((r_lo + r_hi) / 2)
    annular_n_spax.append(n_in)

annular_r_mid = np.array(annular_r_mid)
annular_n_spax = np.array(annular_n_spax)

print(f'Created {len(annular_spectra)} annular bins')
for i, (r, n) in enumerate(zip(annular_r_mid, annular_n_spax)):
    print(f'  Bin {i}: R={r:.2f}", {n} spaxels')

## 5b. Radial binning: PowerBin with radial ordering

In [ ]:
from powerbin import PowerBin

# Use a wider spatial region for PowerBin (same as notebook 01 section 5)
cube_wide = cube[:, 35:75, 30:70]
data_collapsed = np.average(cube_wide, axis=0)
noise_pb = np.median(np.std(cube[:, 28:40, 45:70], axis=(1, 2)))

xp, yp = np.arange(cube_wide.shape[1]), np.arange(cube_wide.shape[2])
xx_pb, yy_pb = np.meshgrid(xp, yp)
xy_pb = np.column_stack([xx_pb.ravel(), yy_pb.ravel()])

signal_flat = data_collapsed.ravel()
noise_flat = np.ones_like(signal_flat) * noise_pb

target_sn = 10

def capacity_spec(index):
    sn = np.sum(signal_flat[index]) / np.sqrt(np.sum(noise_flat[index]**2))
    return sn**2

pow_bins = PowerBin(xy_pb, capacity_spec, target_capacity=target_sn**2)
n_bins_pb = len(pow_bins.npix)

# Compute radial distance for each PowerBin
powerbin_idxs = pow_bins.bin_num
powerbin_idx_map = np.reshape(powerbin_idxs, data_collapsed.shape)

# Get RA/Dec for the wider region
yy_w, xx_w = np.mgrid[35:75, 30:70]
ra_w, dec_w = wcs_ifu.pixel_to_world_values(xx_w.ravel(), yy_w.ravel())
ra_w = ra_w.reshape(40, 40)
dec_w = dec_w.reshape(40, 40)

dra_w = (ra_w - ra_center) * np.cos(np.radians(dec_center)) * 3600
ddec_w = (dec_w - dec_center) * 3600
r_w = np.sqrt(dra_w**2 + ddec_w**2)

pb_r_mean = np.zeros(n_bins_pb)
for b in range(n_bins_pb):
    pb_r_mean[b] = np.mean(r_w[powerbin_idx_map == b])

print(f'PowerBin: {n_bins_pb} bins, target S/N={target_sn}')
print(f'Radial range: {pb_r_mean.min():.2f} - {pb_r_mean.max():.2f} arcsec')

## 6. Run ppxf on radial bins

Fit each radial bin (both annular and PowerBin) with ppxf across
multiple polynomial degrees, using the same approach as notebook 01.

In [ ]:
# ppxf setup (reuse from bootstrap script)
from scripts.bootstrap_ppxf import setup_ppxf_inputs
import importlib
import scripts.bootstrap_ppxf as bp
importlib.reload(bp)

ppxf_inputs = bp.setup_ppxf_inputs(ifu_file, sps_name='fsps')

# We need the ppxf_per_spaxel function from notebook 01
# Inline a simplified version that takes pre-prepared inputs
def ppxf_radial_bin(flux, noise, ppxf_inputs, degrees=np.array([15, 17, 20])):
    """Run ppxf on a single radial bin spectrum."""
    # Rebin to log-wavelength
    log_lam = np.log(lam)
    d_log_lam = (log_lam[-1] - log_lam[0]) / (len(lam) - 1)
    log_lam_new = np.arange(log_lam[0], log_lam[-1] + d_log_lam, d_log_lam)
    flux_r = np.interp(log_lam_new, log_lam, flux)
    noise_r = np.interp(log_lam_new, log_lam, noise)
    lam_r = np.exp(log_lam_new)
    
    mask = (lam_r >= 6500.0) & (lam_r <= 7500.0)
    lam_r, flux_r, noise_r = lam_r[mask], flux_r[mask], noise_r[mask]
    
    galaxy = flux_r / np.median(flux_r)
    noise_norm = np.sqrt(noise_r**2) / np.median(flux_r)
    
    lam_gal = np.copy(lam_r)
    lam_gal *= np.median(util.vac_to_air(lam_gal) / lam_gal)
    
    sps = ppxf_inputs['sps']
    lam_gal_rest = lam_gal / (1 + z_defl)
    goodpixels = util.determine_goodpixels(np.log(lam_gal_rest), [3500, 5000])
    velscale = ppxf_inputs['velscale']
    
    vel_dis = np.zeros(len(degrees))
    error_vdis = np.zeros(len(degrees))
    mean_vel = np.zeros(len(degrees))
    
    for c, deg in enumerate(degrees):
        try:
            pp = ppxf(sps.templates, galaxy, noise_norm, velscale, [0, 300.],
                      goodpixels=goodpixels, plot=False, moments=2, trig=False,
                      degree=deg, lam=lam_gal_rest, lam_temp=sps.lam_temp, mdegree=0)
            vel_dis[c] = pp.sol[1]
            error_vdis[c] = pp.error[1] * np.sqrt(pp.chi2)
            mean_vel[c] = pp.sol[0]
        except Exception:
            vel_dis[c] = np.nan
            error_vdis[c] = np.nan
            mean_vel[c] = np.nan
    
    return {
        'vel_dis': np.nanmean(vel_dis),
        'vel_dis_err': np.nanmean(error_vdis),
        'mean_vel': np.nanmean(mean_vel),
        'vel_dis_all': vel_dis,
    }

In [ ]:
# Run ppxf on annular bins
degrees_fit = np.array([15, 17, 20])

annular_results = []
for i, spec in enumerate(tqdm(annular_spectra, desc='Annular ppxf')):
    result = ppxf_radial_bin(spec, noise_sky, ppxf_inputs, degrees=degrees_fit)
    annular_results.append(result)

sigma_annular = np.array([r['vel_dis'] for r in annular_results])
sigma_err_annular = np.array([r['vel_dis_err'] for r in annular_results])
V_annular = np.array([r['mean_vel'] for r in annular_results])

print('\nAnnular sigma(R):')
for r, s, e, n in zip(annular_r_mid, sigma_annular, sigma_err_annular, annular_n_spax):
    print(f'  R={r:.2f}": sigma={s:.0f} ± {e:.0f} km/s ({n} spaxels)')

In [ ]:
# Run ppxf on PowerBin bins
# Extract binned spectra (same as notebook 01 section 5b)
lam_pb = lam[(lam >= 6500.0) & (lam <= 7500.0)]
mask_pb = (lam >= 6500.0) & (lam <= 7500.0)

fluxes_pb = np.zeros((cube_wide.shape[0], n_bins_pb)).T
for b in range(n_bins_pb):
    fluxes_pb[b] = np.average(cube_wide[:, powerbin_idx_map == b], axis=1)

noise_pb_spec = np.std(cube[:, 28:40, 45:70], axis=(1, 2))

pb_results = []
for b in tqdm(range(n_bins_pb), desc='PowerBin ppxf'):
    result = ppxf_radial_bin(fluxes_pb[b], noise_pb_spec, ppxf_inputs, degrees=degrees_fit)
    pb_results.append(result)

sigma_pb = np.array([r['vel_dis'] for r in pb_results])
sigma_err_pb = np.array([r['vel_dis_err'] for r in pb_results])
V_pb = np.array([r['mean_vel'] for r in pb_results])

## 7. Sigma(R) profile

In [ ]:
# Plot sigma(R) from both methods
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: sigma(R) in arcsec
ax = axes[0]
ax.errorbar(annular_r_mid, sigma_annular, yerr=sigma_err_annular,
            fmt='o-', color='C0', capsize=4, markersize=8, label='Annular rings')
ax.scatter(pb_r_mean, sigma_pb, c='C1', s=60, marker='s', zorder=5, label='PowerBin')

# Integrated value for reference
boot = np.load(f'../results/ppxf_bootstrap_errors_fsps.npz', allow_pickle=True)
sigma_int = np.median(boot['sigma_original'])
ax.axhline(sigma_int, color='gray', ls='--', lw=1, alpha=0.5, label=f'Integrated: {sigma_int:.0f} km/s')

ax.set_xlabel('R (arcsec)')
ax.set_ylabel(r'$\sigma$ (km/s)')
ax.set_title('Velocity Dispersion Profile')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Right: sigma(R) in kpc
ax = axes[1]
ax.errorbar(annular_r_mid * kpc_per_arcsec, sigma_annular, yerr=sigma_err_annular,
            fmt='o-', color='C0', capsize=4, markersize=8, label='Annular rings')
ax.scatter(pb_r_mean * kpc_per_arcsec, sigma_pb, c='C1', s=60, marker='s', zorder=5, label='PowerBin')
ax.axhline(sigma_int, color='gray', ls='--', lw=1, alpha=0.5)

ax.set_xlabel('R (kpc)')
ax.set_ylabel(r'$\sigma$ (km/s)')
ax.set_title('Velocity Dispersion Profile (physical)')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/sigma_radial_profile.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Mean velocity profile (rotation curve)
fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(annular_r_mid, V_annular, fmt='o-', color='C0', capsize=4,
            markersize=8, label='Annular rings')
ax.scatter(pb_r_mean, V_pb, c='C1', s=60, marker='s', zorder=5, label='PowerBin')
ax.axhline(0, color='k', ls='--', lw=0.8)

ax.set_xlabel('R (arcsec)')
ax.set_ylabel('V (km/s)')
ax.set_title('Mean Velocity Profile')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary and comparison

In [ ]:
print('=' * 60)
print('RADIAL VELOCITY DISPERSION PROFILE — AGEL0206')
print('=' * 60)
print(f'\nDeflector center: RA={ra_center:.6f}, Dec={dec_center:.6f}')
print(f'Scale: {kpc_per_arcsec:.2f} kpc/arcsec at z={z_defl}')
print(f'\nAnnular bins: {len(annular_spectra)} rings')
print(f'PowerBin: {n_bins_pb} bins (target S/N={target_sn})')

print(f'\nAnnular sigma(R):')
print(f'{"R (arcsec)":>10} {"R (kpc)":>8} {"sigma":>7} {"err":>5} {"N_spax":>7}')
for r, s, e, n in zip(annular_r_mid, sigma_annular, sigma_err_annular, annular_n_spax):
    print(f'{r:10.2f} {r*kpc_per_arcsec:8.1f} {s:7.0f} {e:5.0f} {n:7d}')

# Save results
np.savez('../results/radial_sigma_profile.npz',
         annular_r_mid=annular_r_mid,
         annular_r_kpc=annular_r_mid * kpc_per_arcsec,
         sigma_annular=sigma_annular,
         sigma_err_annular=sigma_err_annular,
         V_annular=V_annular,
         annular_n_spax=annular_n_spax,
         pb_r_mean=pb_r_mean,
         sigma_pb=sigma_pb,
         sigma_err_pb=sigma_err_pb,
         V_pb=V_pb,
         ra_center=ra_center,
         dec_center=dec_center,
         kpc_per_arcsec=kpc_per_arcsec,
         z_defl=z_defl)
print('\nSaved: ../results/radial_sigma_profile.npz')